In [2]:
import os
from dotenv import load_dotenv
import time
import requests
import numpy as np

import pandas as pd
import json
from math import ceil
load_dotenv()

True

In [3]:
load_dotenv()
# =============================
# OPENAI API SETUP
# =============================
# ⚠️ For safety, prefer setting this in your environment instead of hardcoding
# export OPENAI_API_KEY="your_key_here"
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
OPENAI_URL = "https://api.openai.com/v1/chat/completions"

if not OPENAI_API_KEY:
    raise EnvironmentError("OPENAI_API_KEY is not set")


In [5]:
# test = "What is the capital of France?"
# =============================
def call_openai_api(messages, model="gpt-5.4-mini", temperature=0, max_completion_tokens=128000):
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {OPENAI_API_KEY}"
    }
    payload = {
        "model": model, 
        "messages": messages,
        "temperature": temperature,
        "max_completion_tokens": max_completion_tokens
    }   
    response = requests.post(OPENAI_URL, headers=headers, json=payload)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.text}")
        return None
# =============================

# call_openai_api([{"role": "user", "content": "What is the capital of France?"}])


In [6]:
# # =============================
# # CATEGORY DEFINITIONS (Compact LLM Version)
# # =============================
# CATEGORY_DEFINITIONS = {
#     "lokasi": "Keluar karena pindah lokasi geografis secara eksplisit (pindah kota, luar kota, luar negeri/relokasi).",

#     "perbedaan_musim_kehidupan": "Keluar karena perubahan fase hidup jangka panjang (menikah, punya anak, jenjang studi baru, perubahan karier besar). Bukan konflik jadwal rutin.",

#     "tidak_ada_respon": "Tidak atau minim respons saat dihubungi (tidak balas, sulit dihubungi, tidak ada kabar).",

#     "tertanam_di_gereja_lain": "Memilih tetap tertanam atau aktif di gereja lain, bukan di JPCC.",

#     "waktu_tidak_sesuai": "Benturan jadwal atau komitmen waktu rutin (jam kerja, shift, pulang malam, jadwal kuliah).",

#     "alasan_DATE": "Masalah atau kondisi terkait kelompok DATE (tidak cocok, pindah DATE, konflik leader/member, DATE close/bubar).",

#     "perbedaan_umur": "Tidak cocok karena perbedaan atau rentang usia dalam kelompok.",

#     "Admin": "Kesalahan administratif atau perubahan data yang bukan keputusan pribadi anggota(human error, new comer, probation, false positive).",

#     "others": "Alasan tidak jelas, wafat, terlalu singkat, ambigu, atau tidak termasuk kategori lain."
# }

In [7]:
# =============================
# CATEGORY DEFINITIONS (New Version)
# =============================
CATEGORY_DEFINITIONS = {
    "lokasi": "Keluar karena pindah lokasi geografis secara eksplisit (pindah kota, luar kota, luar negeri/relokasi).",

    "perbedaan_musim_kehidupan": "Keluar karena perubahan fase hidup jangka panjang (menikah, punya anak, jenjang studi baru, perubahan karier besar). Bukan konflik jadwal rutin.",

    "tidak_ada_respon": "Tidak atau minim respons saat dihubungi, Missing in Action (tidak balas, sulit dihubungi, tidak pernah hadir).",

    "tertanam_di_gereja_lain": "Memilih tetap tertanam atau aktif di gereja lain, bukan di JPCC.",

    "waktu_tidak_sesuai": "Benturan jadwal atau komitmen waktu rutin (jam kerja, shift, pulang malam, jadwal kuliah).",

    "alasan_DATE": "Masalah atau kondisi terkait kelompok DATE (tidak cocok, beda usia, pindah DATE, konflik leader/member, DATE close/bubar).",

    "Admin": "Kesalahan administratif atau perubahan data yang bukan keputusan pribadi anggota (human error, new comer, probation, false positive).",

    "others": "Alasan tidak jelas, wafat, terlalu singkat, ambigu, atau tidak termasuk kategori lain."
}   

In [ ]:
# =============================
# BATCH CLASSIFIER (GPT-5.4 MINI → JSON)
# =============================

def classify_batch_with_gpt54_mini(texts: list[str]) -> list[dict]:
    """
    Classify multiple texts in ONE request using GPT-5.4 mini.
    Returns a parsed JSON array (list of dicts).
    """

    prompt = f"""
You are a data annotation system.

Categories:
{json.dumps(CATEGORY_DEFINITIONS, ensure_ascii=False)}

Rules:
- Choose EXACTLY ONE category per text.
- Label MUST match one of the category keys.
- If relocation is explicitly mentioned → choose "lokasi".
- If routine schedule conflict → choose "waktu_tidak_sesuai".
- If long-term life phase change → choose "perbedaan_musim_kehidupan".
- If related to DATE group condition/conflict → choose "alasan_DATE".
- If unclear or insufficient information → choose "others".
- Do not infer beyond the text.

Return a JSON array (same order as input).
Each object must contain:
- final_label (string)
- confidence (0–100 integer)
- keywords (Max 3 short keywords)

Output ONLY valid JSON. No explanations.

Texts:
{json.dumps(texts, ensure_ascii=False)}
"""

    payload = {
        "model": "gpt-5.4-mini",
        "messages": [
            {"role": "system", "content": "You are a precise classification engine."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0, # Deterministic output for consistent classification
    }

    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json",
    }

    for attempt in range(10):
        response = requests.post(
            OPENAI_URL,
            json=payload,
            headers=headers,
            timeout=60,
        )

        if response.status_code == 429:
            wait_time = 5 * (attempt + 1)
            print(f"Rate limited. Sleeping {wait_time}s...")
            time.sleep(wait_time)
            continue

        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"]

        try:
            return json.loads(content)
        except json.JSONDecodeError:
            raise ValueError("Model did not return valid JSON")

    raise RuntimeError("Failed after retries due to rate limiting")


In [7]:
# split data df menjadi 5 file variable yang berbeda
df = pd.read_csv("freetext.csv")
df_split = np.array_split(df, 5)

for i, split_df in enumerate(df_split):
    split_df.to_csv(f"freetext_split_{i+1}.csv", index=False) 


c:\Users\Jovan\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [15]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    df = pd.read_csv("freetext_split_1.csv")
    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        # Batch label format: "batch : X of Y row: Z" (row is 1-based within batch)
        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_gpt54_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch")

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            })

    pd.DataFrame(results).to_csv("labeled_resultsGPT1.csv", index=False)
    print("Saved to labeled_resultsGPT1.csv")
    

if __name__ == "__main__":
    run_pipeline(batch_size=10)

Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Processing batch : 17 of 48
Processing batch : 18 of 48
Processing batch : 19 of 48
Processing batch : 20 of 48
Processing batch : 21 of 48
Processing batch : 22 of 48
Processing batch : 23 of 48
Processing batch : 24 of 48
Processing batch : 25 of 48
Processing batch : 26 of 48
Processing batch : 27 of 48
Processing batch : 28 of 48
Processing batch : 29 of 48
Processing batch : 30 of 48
Processing batch : 31 of 48
Processing batch : 32 of 48
Processing batch : 33 of 48
Processing batch : 34 of 48
Processing batch : 35 of 48
Processing batch : 36 of 48
P

In [16]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    df = pd.read_csv("freetext_split_2.csv")
    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        # Batch label format: "batch : X of Y row: Z" (row is 1-based within batch)
        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_gpt54_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch")

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            })

    pd.DataFrame(results).to_csv("labeled_resultsGPT2.csv", index=False)
    print("Saved to labeled_resultsGPT2.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)

Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Processing batch : 17 of 48
Processing batch : 18 of 48
Processing batch : 19 of 48
Processing batch : 20 of 48
Processing batch : 21 of 48
Processing batch : 22 of 48
Processing batch : 23 of 48
Processing batch : 24 of 48
Processing batch : 25 of 48
Processing batch : 26 of 48
Processing batch : 27 of 48
Processing batch : 28 of 48
Processing batch : 29 of 48
Processing batch : 30 of 48
Processing batch : 31 of 48
Processing batch : 32 of 48
Processing batch : 33 of 48
Processing batch : 34 of 48
Processing batch : 35 of 48
Processing batch : 36 of 48
P

In [17]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    df = pd.read_csv("freetext_split_3.csv")
    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        # Batch label format: "batch : X of Y row: Z" (row is 1-based within batch)
        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_gpt54_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch")

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            })

    pd.DataFrame(results).to_csv("labeled_resultsGPT3.csv", index=False)
    print("Saved to labeled_resultsGPT3.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)

Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Processing batch : 17 of 48
Processing batch : 18 of 48
Processing batch : 19 of 48
Processing batch : 20 of 48
Processing batch : 21 of 48
Processing batch : 22 of 48
Processing batch : 23 of 48
Processing batch : 24 of 48
Processing batch : 25 of 48
Processing batch : 26 of 48
Processing batch : 27 of 48
Processing batch : 28 of 48
Processing batch : 29 of 48
Processing batch : 30 of 48
Processing batch : 31 of 48
Processing batch : 32 of 48
Processing batch : 33 of 48
Processing batch : 34 of 48
Processing batch : 35 of 48
Processing batch : 36 of 48
P

In [18]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    df = pd.read_csv("freetext_split_4.csv")
    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        # Batch label format: "batch : X of Y row: Z" (row is 1-based within batch)
        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_gpt54_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch")

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            })

    pd.DataFrame(results).to_csv("labeled_resultsGPT4.csv", index=False)
    print("Saved to labeled_resultsGPT4.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)

Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Processing batch : 17 of 48
Processing batch : 18 of 48
Processing batch : 19 of 48
Processing batch : 20 of 48
Processing batch : 21 of 48
Processing batch : 22 of 48
Processing batch : 23 of 48
Processing batch : 24 of 48
Processing batch : 25 of 48
Processing batch : 26 of 48
Processing batch : 27 of 48
Processing batch : 28 of 48
Processing batch : 29 of 48
Processing batch : 30 of 48
Processing batch : 31 of 48
Processing batch : 32 of 48
Processing batch : 33 of 48
Processing batch : 34 of 48
Processing batch : 35 of 48
Processing batch : 36 of 48
P

In [19]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    df = pd.read_csv("freetext_split_5.csv")
    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        # Batch label format: "batch : X of Y row: Z" (row is 1-based within batch)
        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_gpt54_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch")

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            })

    pd.DataFrame(results).to_csv("labeled_resultsGPT5.csv", index=False)
    print("Saved to labeled_resultsGPT5.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)

Processing batch : 1 of 48
Processing batch : 2 of 48
Processing batch : 3 of 48
Processing batch : 4 of 48
Processing batch : 5 of 48
Processing batch : 6 of 48
Processing batch : 7 of 48
Processing batch : 8 of 48
Processing batch : 9 of 48
Processing batch : 10 of 48
Processing batch : 11 of 48
Processing batch : 12 of 48
Processing batch : 13 of 48
Processing batch : 14 of 48
Processing batch : 15 of 48
Processing batch : 16 of 48
Processing batch : 17 of 48
Processing batch : 18 of 48
Processing batch : 19 of 48
Processing batch : 20 of 48
Processing batch : 21 of 48
Processing batch : 22 of 48
Processing batch : 23 of 48
Processing batch : 24 of 48
Processing batch : 25 of 48
Processing batch : 26 of 48
Processing batch : 27 of 48
Processing batch : 28 of 48
Processing batch : 29 of 48
Processing batch : 30 of 48
Processing batch : 31 of 48
Processing batch : 32 of 48
Processing batch : 33 of 48
Processing batch : 34 of 48
Processing batch : 35 of 48
Processing batch : 36 of 48
P

# Check Data

In [ ]:
df = pd.read_csv("free_text.csv")
df

FileNotFoundError: [Errno 2] No such file or directory: 'free_text.csv'

In [ ]:
df2 = pd.read_csv("labeled_resultsGPT3.csv")
df2

,delete_reason,batch,Final label,Confidence,Keywords
0,sorry ini harusnya jd New Comer dulu,1 (1 of 10),Admin,80%,"New Comer, status, perubahan, data"
1,"sudah pindah rumah & gereja ke Bandung, jadi t...",1 (1 of 10),lokasi,95%,"pindah rumah, Bandung, tidak datang, gereja"
2,Pindah gereja,1 (1 of 10),lokasi,90%,"pindah, gereja, lokasi, perubahan"
3,Tertanam di gereja lain,1 (1 of 10),masih_tertanam_di_gereja_lain,95%,"tertanam, gereja lain, tidak di JPCC, keputusan"
4,meninggal dunia,1 (1 of 10),Admin,85%,"meninggal dunia, status, data, pengubahan"
...,...,...,...,...,...
95,mencari DATE yang lebih terjangkau dari tempat...,10 (10 of 10),lokasi,80%,"mencari, terjangkau, tempat kerja, lokasi"
96,pindah ke luar negeri,10 (10 of 10),lokasi,95%,"pindah, luar negeri, lokasi, domisili"
97,Karena belum moment yg tepat untuk diterima di...,10 (10 of 10),waktu_tidak_sesuai,85%,"belum, moment, tepat, date"
98,karena kepenuhan (jadi hanya pasangannya saja ...,10 (10 of 10),grup_penuh,90%,"kepenuhan, date, member, transfer"


In [10]:
# =============================
# PIPELINE (BATCHED → STRUCTURED CSV)
# =============================

def run_pipeline(batch_size: int = 10):
    df = pd.read_csv("consistencydata.csv")
    results = []

    num_batches = ceil(len(df) / batch_size)

    for batch_idx in range(num_batches):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(df))
        batch = df.iloc[start:end]

        # Batch label format: "batch : X of Y row: Z" (row is 1-based within batch)
        batch_label_base = f"batch : {batch_idx + 1} of {num_batches}"
        print(f"Processing {batch_label_base}")

        try:
            outputs = classify_batch_with_gpt54_mini(
                batch["delete_reason"].tolist()
            )

            if len(outputs) != len(batch):
                raise ValueError("Output length mismatch")

        except Exception as e:
            print("Batch error:", e)
            outputs = [{} for _ in range(len(batch))]

        for i, (_, row) in enumerate(batch.iterrows()):
            out = outputs[i]
            results.append({
                "delete_reason": row["delete_reason"],
                "batch": f"{batch_label_base} row: {i + 1}",
                "Final label": out.get("final_label", ""),
                "Confidence": f"{out.get('confidence', '')}%" if out else "",
                "Keywords": ", ".join(out.get("keywords", [])) if out else "",
            })

    pd.DataFrame(results).to_csv("ConsistencyGPT.csv", index=False)
    print("Saved to ConsistencyGPT.csv")


if __name__ == "__main__":
    run_pipeline(batch_size=10)

Processing batch : 1 of 25
Processing batch : 2 of 25
Processing batch : 3 of 25
Processing batch : 4 of 25
Processing batch : 5 of 25
Processing batch : 6 of 25
Processing batch : 7 of 25
Processing batch : 8 of 25
Processing batch : 9 of 25
Processing batch : 10 of 25
Processing batch : 11 of 25
Processing batch : 12 of 25
Processing batch : 13 of 25
Processing batch : 14 of 25
Processing batch : 15 of 25
Processing batch : 16 of 25
Processing batch : 17 of 25
Processing batch : 18 of 25
Processing batch : 19 of 25
Processing batch : 20 of 25
Processing batch : 21 of 25
Processing batch : 22 of 25
Processing batch : 23 of 25
Processing batch : 24 of 25
Processing batch : 25 of 25
Saved to ConsistencyGPT.csv
